# Laboratorio 5. Regularización Rigde y Lasso

## Carga y preparación de los datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Utilizaremos el conjunto de datos **Diabetes** incluido en `sklearn`. Contiene información de 442 pacientes y 10 variables predictoras numéricas relacionadas con características clínicas y demográficas.

La variable de respuesta representa una **medida cuantitativa de progresión de la enfermedad un año después de la medición inicial**.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)

print(data.DESCR)

**Nota**: Las variables s1–s6 corresponden a seis mediciones obtenidas del suero sanguíneo: colesterol total (s1), LDL (s2), HDL (s3), colesterol total/HDL (s4), triglicéridos (s5) y glucosa (s6).

In [ ]:
X = data.data
y = data.target

X.info()

In [ ]:
X.head()

In [ ]:
corr = X.corr()

plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}",
                 ha="center", va="center", size=8, c="0.2")

plt.title("Correlación entre predictores")
plt.show()

Se observa una **alta correlación positiva entre `s1` y `s2`** y una **alta correlación negativa entre `s3` y `s4`**. También se aprecia una correlación positiva moderada entre `s2` y `s4`. Para el resto de los pares, las correlaciones lineales son débiles o moderadas. **Sin embargo, una baja correlación entre pares de predictores no implica necesariamente ausencia de multicolinealidad**, ya que una variable puede estar fuertemente relacionada con una **combinación lineal de varias de las demás variables**, relación que no puede detectarse únicamente mediante la matriz de correlaciones por pares.

Recordemos que la **multicolinealidad puede provocar coeficientes de regresión de magnitud muy grande y altamente sensibles a pequeñas variaciones en los datos**, haciendo que las estimaciones de los coeficientes y, potencialmente, las predicciones del modelo sean inestables. Para reducir estos efectos, utilizaremos **regresión regularizada**, donde se introduce una penalización sobre la magnitud de los coeficientes con el objetivo de controlar su crecimiento y mejorar la estabilidad del modelo.

## Implementación simple


Primero implementaremos un modelo de regresion lineal sin regularización para poder observar **qué pasa con los coeficientes** cuando agregamos una penalización.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso

lm = LinearRegression()
lm.fit(X_train, y_train)

pd.Series(lm.coef_, index=X.columns)

Los predictores altamente correlacionados presentan algunos de los **coeficientes de mayor magnitud**, particularmente `s1` (-918.50) y `s2` (508.26). Esto es consistente con la presencia de **multicolinealidad**, que puede producir coeficientes grandes e inestables.

Para reducir la magnitud de los coeficientes utilizaremos **Ridge Regression** como estrategia de regularización. Por el momento, utilizaremos $\alpha = 1$.

In [ ]:
ridge = Ridge(alpha=1)
ridge.fit(X_train, y_train)

pd.Series(ridge.coef_, index=X.columns)

Como segunda estrategia de regularización utilizaremos **Lasso Regression**, manteniendo el mismo nivel de penalización, $\alpha = 1$.

In [ ]:
lasso = Lasso(alpha=1)
lasso.fit(X_train, y_train)

pd.Series(lasso.coef_, index=X.columns)

In [ ]:
coef = pd.DataFrame({
    "Variable": X.columns,
    "Lineal": lm.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_
})

coef

La regresión **Ridge** reduce considerablemente la magnitud de los coeficientes, especialmente en variables como `s1` y `s2`, aunque mantiene todas las variables en el modelo. Por otro lado, **Lasso** lleva varios coeficientes exactamente a cero, conservando principalmente `bmi`, `bp` y `s5`. Esto muestra la diferencia fundamental entre ambas estrategias: **Ridge reduce los coeficientes, mientras que Lasso además puede realizar selección de variables**.

## Paths de regularización

Hasta ahora hemos utilizado un **coeficiente de regularización arbitrario** ($\alpha = 1$). Observemos ahora el **efecto del parámetro de regularización** sobre la magnitud de los coeficientes conforme incrementamos el valor de $\alpha$.

In [ ]:
# Ajustamos regresión Ridge para cada valor de alpha

alphas = np.linspace(0, 10, 100)

ridge_coefs = []

for alpha in alphas:

    model = Ridge(alpha=alpha)
    model.fit(X, y)

    ridge_coefs.append(model.coef_)

ridge_coefs = np.array(ridge_coefs)

In [ ]:
for i, variable in enumerate(X.columns):
    plt.plot(alphas, ridge_coefs[:, i], label=variable)

plt.xlabel("alpha")
plt.ylabel("Coeficiente")
plt.title("Paths de regularización - Ridge")

plt.legend(bbox_to_anchor=(1.05, 1))
plt.show()

A medida que aumenta $\lambda$, los coeficientes de **Ridge disminuyen progresivamente en magnitud y convergen hacia cero**. El efecto es especialmente notable en los coeficientes inicialmente más grandes, mostrando cómo la regularización controla estimaciones extremas. Sin embargo, **Ridge no lleva los coeficientes exactamente a cero**, sino que reduce gradualmente su contribución.

In [ ]:
# Ajustamos regresión Lasso para cada valor de alpha

alphas = np.linspace(0, 3, 100)

lasso_coefs = []

for alpha in alphas:

    model = Lasso(alpha=alpha, max_iter=10000)
    model.fit(X, y)

    lasso_coefs.append(model.coef_)

lasso_coefs = np.array(lasso_coefs)

In [ ]:
for i, variable in enumerate(X.columns):
    plt.plot(alphas, lasso_coefs[:, i], label=variable)

plt.xlabel("alpha")
plt.ylabel("Coeficiente")
plt.title("Paths de regularización - Lasso")

plt.legend(bbox_to_anchor=(1.05, 1))
plt.show()

A diferencia de Ridge, **Lasso puede llevar los coeficientes exactamente a cero** conforme aumenta $\lambda$, eliminando predictores del modelo. En este caso, una penalización de aproximadamente $\lambda = 2.5$ es suficiente para que **todos los coeficientes sean cero**, mostrando que una regularización excesiva puede eliminar completamente la capacidad predictiva del modelo.

## Validación cruzada

Hasta ahora elegimos $\alpha$ arbitrariamente. En un problema real, este valor debe considerarse un **hiperparámetro** y seleccionarse utilizando datos que no hayan sido utilizados para ajustar el modelo.

Para ello utilizaremos **validación cruzada**, evaluando diferentes valores de $\alpha$ y seleccionando aquel que produzca el **menor error cuadrático medio (MSE) promedio en validación**.

## Regresión Ridge

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# Valores candidatos para el parámetro de regularización
alphas = np.linspace(1e-10, 1e-2, 100)

# División de los datos en 5 folds para validación cruzada
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# MSE promedio de validación para cada valor de alpha
mse_alpha = []

for alpha in alphas:

    # MSE de validación obtenido en cada fold
    mse_folds = []

    for train_idx, val_idx in kf.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Modelo Ridge con el parámetro de regularización actual
        model = Ridge(alpha=alpha)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)

        mse_folds.append(mean_squared_error(y_val, y_pred))

    # MSE promedio de los folds
    mse_alpha.append(np.mean(mse_folds))

In [ ]:
plt.plot(alphas, mse_alpha)

plt.xlabel(r"$\alpha$")
plt.ylabel("MSE promedio de validación")
plt.title("Selección del parámetro de regularización - Ridge")

plt.show()

Debemos seleccionar el valor de $\alpha$ que produzca el **menor MSE promedio de validación**, ya que este representa el nivel de regularización que ofrece el mejor desempeño sobre los datos de validación.

In [ ]:
best_alpha = alphas[np.argmin(mse_alpha)]
best_mse = np.min(mse_alpha)

plt.plot(alphas, mse_alpha)
plt.scatter(best_alpha, best_mse)

plt.xlabel(r"$\alpha$")
plt.ylabel("MSE promedio de validación")
plt.title("Selección del parámetro de regularización - Ridge")

plt.show()

print("Mejor alpha:", best_alpha)

Utilizamos el alpha con menor MSE promedio de validación para ajustar el modelo final.

In [ ]:
from sklearn.metrics import mean_squared_error

ridge = Ridge(alpha=best_alpha)

ridge.fit(X_train, y_train)

pd.Series(ridge.coef_, index=X.columns)

## Regresión Lasso

In [ ]:
# Valores candidatos para el parámetro de regularización
alphas = np.linspace(1e-5, 1e-2, 100)

# División de los datos en 5 folds para validación cruzada
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# MSE promedio de validación para cada valor de alpha
mse_alpha_lasso = []

for alpha in alphas:

    # MSE de validación obtenido en cada fold
    mse_folds = []

    for train_idx, val_idx in kf.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Modelo Lasso con el parámetro de regularización actual
        model = Lasso(alpha=alpha)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)

        mse_folds.append(mean_squared_error(y_val, y_pred))

    # MSE promedio de los folds
    mse_alpha_lasso.append(np.mean(mse_folds))

In [ ]:
plt.plot(alphas, mse_alpha_lasso)

plt.xlabel(r"$\alpha$")
plt.ylabel("MSE promedio de validación")
plt.title("Selección del parámetro de regularización - Lasso")

plt.show()

In [ ]:
best_alpha_lasso = alphas[np.argmin(mse_alpha_lasso)]

print("Mejor alpha:", best_alpha_lasso)

In [ ]:
from sklearn.metrics import mean_squared_error

lasso = Lasso(alpha=best_alpha)

lasso.fit(X_train, y_train)

pd.Series(lasso.coef_, index=X.columns)

In [ ]:
# MSE de regresión lineal
mse_lineal = [
    mean_squared_error(y_train, lm.predict(X_train)),
    mean_squared_error(y_test, lm.predict(X_test))
]

# MSE de Ridge
mse_ridge = [
    mean_squared_error(y_train, ridge.predict(X_train)),
    mean_squared_error(y_test, ridge.predict(X_test))
]

# MSE de Lasso
mse_lasso = [
    mean_squared_error(y_train, lasso.predict(X_train)),
    mean_squared_error(y_test, lasso.predict(X_test))
]


# Tabla comparativa
mse_comparison = pd.DataFrame({
    "Lineal": mse_lineal,
    "Ridge": mse_ridge,
    "Lasso": mse_lasso
}, index=["Train", "Test"])

mse_comparison


In [ ]:
# Diferencia del MSE respecto a regresión lineal
delta_ridge = np.array(mse_ridge) - np.array(mse_lineal)
delta_lasso = np.array(mse_lasso) - np.array(mse_lineal)

x = np.arange(2)
w = 0.3

plt.bar(x - w/2, delta_ridge, width=w, label="Ridge")
plt.bar(x + w/2, delta_lasso, width=w, label="Lasso")

plt.axhline(0, linewidth=1)

plt.xticks(x, ["Train", "Test"])
plt.ylabel(r"$\Delta$ MSE respecto a Lineal")
plt.title("Cambio en MSE por regularización")
plt.legend()

plt.show()

La regularización **empeora ligeramente el MSE de entrenamiento**, lo que se observa como una diferencia positiva respecto al modelo lineal. Sin embargo, **mejora el MSE en datos no observados**, reflejado por una diferencia negativa. Es decir, sacrificamos parte del ajuste en entrenamiento a cambio de una **mejor capacidad de generalización**.